# 02 — Preprocessing
Clean the data, keep 12 audio features across 8 acoustically distinct genres (2,000 samples each), encode labels, scale, and split.
Feature importances from notebook 03 will guide any future cuts.

In [ ]:
# Cell 1 — Imports
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# Cell 2 — Load full dataset (no genre filtering this time)
df = pd.read_csv("../data/SpotifySongs.csv", index_col=0)
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Genres: {df['track_genre'].nunique()} unique")

In [ ]:
# Cell 3 — Clean: drop duplicates and null rows
# Duplicates can bias the model; nulls will crash scikit-learn
before = len(df)

df = df.drop_duplicates()
dupes_removed = before - len(df)

df = df.dropna()
nulls_removed = before - dupes_removed - len(df)

print(f"Duplicates removed : {dupes_removed:,}")
print(f"Null rows removed  : {nulls_removed:,}")
print(f"Rows remaining     : {len(df):,}")

In [ ]:
# Cell 4 — Feature selection: 12 of the 15 numeric features
# Dropped: explicit (content label, not an audio property),
#          key (cyclical ordinal — needs special encoding to be useful),
#          time_signature (near-zero variance, most tracks are 4/4)
ALL_FEATURES = [
    'popularity',
    'duration_ms',
    'mode',
    'danceability',
    'energy',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'loudness',
    'speechiness',
    'acousticness',
]
TARGET = 'track_genre'

df = df[ALL_FEATURES + [TARGET]]
print(f"Columns kept: {df.columns.tolist()}")
df.head()

In [ ]:
# We narrow to 8 genres chosen for maximal acoustic separation.
# Each one occupies a distinct region of the audio feature space —
# this is what lets the model tell them apart reliably.
GENRES_8 = [
    'classical',    # high acousticness, low energy, low speechiness
    'hip-hop',      # high speechiness, high danceability
    'electronic',   # high energy, high danceability, low acousticness
    'country',      # moderate acousticness, high valence, mid tempo
    'jazz',         # moderate acousticness, low speechiness, mid energy
    'black-metal',  # very high energy, high loudness, low valence
    'latin',        # high danceability, high valence, mid-high energy
    'pop',          # balanced across most features
]

df = df[df['track_genre'].isin(GENRES_8)]
print(f"Rows remaining : {len(df)}")
print(f"Genres         : {df['track_genre'].nunique()}")
print(df['track_genre'].value_counts())

In [ ]:
SAMPLES_PER_GENRE = 2000

sampled_dfs = []
for genre in df['track_genre'].unique():
    genre_df = df[df['track_genre'] == genre]
    sampled_dfs.append(genre_df.sample(min(len(genre_df), SAMPLES_PER_GENRE), random_state=42))

df = pd.concat(sampled_dfs).reset_index(drop=True)
print(df['track_genre'].value_counts())

In [ ]:
print(df.columns.tolist())
print(df['track_genre'].unique())

In [ ]:
# Cell 5 — Encode target labels
# LabelEncoder turns genre strings into integers (e.g. 'acoustic' → 0, 'afrobeat' → 1, ...)
# We save the encoder so we can convert predictions back to genre names later
le = LabelEncoder()
y = le.fit_transform(df[TARGET])

print(f"Total classes: {len(le.classes_)}")
print("\nLabel encoding map (first 20):")
for i, genre in enumerate(le.classes_[:20]):
    print(f"  {i:>3} → {genre}")
print("  ...")

In [ ]:
# Cell 6 — Train/test split, then scale features
# IMPORTANT: scaler is fit ONLY on training data to prevent data leakage.
# Leakage = the model accidentally "sees" test data during training → inflated scores.
X = df[ALL_FEATURES].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # keeps genre proportions equal in both splits
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # fit + transform on train
X_test  = scaler.transform(X_test)       # transform only on test

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

In [ ]:
# Cell 7 — Verify class distribution in each split
# Stratify should keep counts balanced; this confirms it worked
train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts  = pd.Series(y_test).value_counts().sort_index()

dist = pd.DataFrame({
    'genre'      : le.classes_,
    'train_count': train_counts.values,
    'test_count' : test_counts.values
})
print(dist.to_string(index=False))

In [ ]:
# Cell 8 — Save processed arrays, scaler, and encoder
# Notebooks 03 and 04 load these files instead of re-running preprocessing
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

np.save("../data/processed/X_train.npy", X_train)
np.save("../data/processed/X_test.npy",  X_test)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/y_test.npy",  y_test)

joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(le,     "../models/label_encoder.pkl")

print("Saved:")
print("  data/processed/X_train.npy")
print("  data/processed/X_test.npy")
print("  data/processed/y_train.npy")
print("  data/processed/y_test.npy")
print("  models/scaler.pkl")
print("  models/label_encoder.pkl")